# Panel data: fixed and random effects: lab notebook

**Statistical Modelling and Forecasting, Multivariate and advanced methods**

Run the cells from top to bottom (Runtime > Run all). NumPy, pandas, SciPy, statsmodels and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # keeps the output tidy

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats

rng = np.random.default_rng(4)

def make_panel(N=40, T=6, corr=0.7, su=4.0, beta=1.5):
    m = 2 * rng.standard_normal(N)
    u = su * (corr * m / 2 + np.sqrt(1 - corr ** 2) * rng.standard_normal(N))
    rows = []
    for i in range(N):
        for t in range(T):
            x = m[i] + rng.standard_normal()
            rows.append((i, t, x, beta * x + u[i] + rng.standard_normal()))
    return pd.DataFrame(rows, columns=["unit", "t", "x", "y"])

def estimate(df):
    N, T = df.unit.nunique(), df.t.nunique()
    pooled = sm.OLS(df.y, sm.add_constant(df.x)).fit()
    xm = df.groupby("unit").x.transform("mean"); ym = df.groupby("unit").y.transform("mean")
    xd, yd = df.x - xm, df.y - ym
    fe = (xd * yd).sum() / (xd ** 2).sum()
    s2e = ((yd - fe * xd) ** 2).sum() / (N * T - N - 1)
    be = sm.OLS(ym.groupby(df.unit).first(), sm.add_constant(xm.groupby(df.unit).first())).fit()
    s2u = max(0, be.scale - s2e / T)
    th = 1 - np.sqrt(s2e / (s2e + T * s2u))
    xq, yq = df.x - th * xm, df.y - th * ym
    re = sm.OLS(yq, sm.add_constant(xq)).fit()
    vfe = s2e / (xd ** 2).sum(); vre = s2e / ((xq - xq.mean()) ** 2).sum()
    H = (fe - re.params.iloc[1]) ** 2 / (vfe - vre) if vfe > vre else np.nan
    return dict(pooled=pooled.params["x"], fe=fe, re=re.params.iloc[1], theta=th, s2e=s2e, s2u=s2u, H=H, p=stats.chi2.sf(H, 1))

### 1. One panel: pooled, fixed effects and random effects (true slope 1.5)

In [ ]:
# 1. One panel: pooled, fixed effects and random effects (true slope 1.5)
df = make_panel()
r = estimate(df)
print("slopes: pooled", round(r["pooled"], 3), "| FE", round(r["fe"], 3), "| RE", round(r["re"], 3))
print("sigma_e^2:", round(r["s2e"], 3), "| sigma_u^2:", round(r["s2u"], 3), "| theta:", round(r["theta"], 3))
print("Hausman chi2 =", round(r["H"], 2), "p =", round(r["p"], 4))

### 2. Check the FE slope against least squares with unit dummies (LSDV)

In [ ]:
# 2. Check the FE slope against least squares with unit dummies (LSDV)
D = pd.get_dummies(df.unit, prefix="u", dtype=float)
lsdv = sm.OLS(df.y, pd.concat([df.x, D], axis=1)).fit()
print("\nLSDV slope:", round(lsdv.params["x"], 4), "| within slope:", round(r["fe"], 4))

### 3. Bias over many samples, with the unit effects correlated with x

In [ ]:
# 3. Bias over many samples, with the unit effects correlated with x
res = pd.DataFrame([estimate(make_panel()) for _ in range(150)])
print("\nmean slope over 150 panels (true 1.5):")
print(res[["pooled", "fe", "re"]].mean().round(3).to_string())
print("Hausman rejects at 5% in", round((res.p < 0.05).mean() * 100), "% of panels")

### 4. Independent effects: all estimators are fine and RE is preferred

In [ ]:
# 4. Independent effects: all estimators are fine and RE is preferred
res0 = pd.DataFrame([estimate(make_panel(corr=0.0)) for _ in range(150)])
print("\nmean slope when effects are unrelated to x:")
print(res0[["pooled", "fe", "re"]].mean().round(3).to_string())
print("Hausman rejects in", round((res0.p < 0.05).mean() * 100), "% of panels (about 5% expected)")
print("sd of slope: FE", round(res0.fe.std(), 3), "| RE", round(res0.re.std(), 3), "-> RE is more efficient")

## Expected output

Running all cells above should print:

```
slopes: pooled 2.818 | FE 1.48 | RE 1.589
sigma_e^2: 0.824 | sigma_u^2: 9.254 | theta: 0.879
Hausman chi2 = 46.1 p = 0.0

LSDV slope: 1.4795 | within slope: 1.4795

mean slope over 150 panels (true 1.5):
pooled    2.603
fe        1.501
re        1.621
Hausman rejects at 5% in 99 % of panels

mean slope when effects are unrelated to x:
pooled    1.480
fe        1.493
re        1.492
Hausman rejects in 5 % of panels (about 5% expected)
sd of slope: FE 0.075 | RE 0.071 -> RE is more efficient
```

## Practice exercises

Try these in the cells below.

1. Set the correlation between the unit effects and x to 0. All three estimates agree. Why?
2. Raise the correlation to 0.8. Which estimators are biased? What does the Hausman statistic say?
3. Increase the variance of the unit effects and watch θ move towards 1.

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
